# Network Intrusion Detection — PySpark Data Preprocessing

**Dataset:** CICIDS 2017 (2.8 M network flows, 79 features, 15 attack categories)

Each major step saves a **Parquet checkpoint** (Spark's equivalent of `.pkl`).  
A lightweight **`.pkl` config file** is also written so the pipeline can be inspected without Spark.

| Step | Description | Checkpoint |
|------|-------------|------------|
| 1 | Setup & Configuration | — |
| 2 | Data Loading | — |
| 3 | Exploratory Data Analysis | — |
| 4.1 | Strip column-name whitespace | — |
| 4.2 | Remove duplicate rows | `checkpoints/01_dedup/` |
| 4.3 | Audit missing values | — |
| 4.4 | Replace Inf / NaN | — |
| 4.5 | Fix label encoding | — |
| 4.6 | Drop irrelevant columns | — |
| 4.7 | Standardize label names | `checkpoints/02_cleaned/` + `pipeline_step2.pkl` |
| 5 | Class balancing | `checkpoints/03_balanced/` + `pipeline_step3.pkl` |
| 6 | Save final dataset | `data/clean/` + `pipeline_final.pkl` |

---
## 1. Setup & Configuration

In [4]:
import time
import os
import pickle
import shutil
import numpy as np
import matplotlib.pyplot as plt

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col, when, trim, regexp_replace,
    isnan, sum as spark_sum
)
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType, FloatType
)

try:
    spark.stop()
except Exception:
    pass


spark = (
    SparkSession.builder
    .appName("CICIDS2017-Preprocessing")
    .master("local[24]")
    .config("spark.driver.memory", "180g")
    .config("spark.driver.maxResultSize", "20g")
    .config("spark.sql.shuffle.partitions", "96")
    .config("spark.default.parallelism", "96")
    .config("spark.memory.fraction", "0.75")
    .config("spark.memory.storageFraction", "0.3")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.sql.adaptive.skewJoin.enabled", "true")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

print(f"Driver memory      : {spark.conf.get('spark.driver.memory')}")
print(f"Shuffle partitions : {spark.conf.get('spark.sql.shuffle.partitions')}")
print(f"Master             : {spark.conf.get('spark.master')}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/03 00:17:20 WARN Utils: Your hostname, ml-ai-ubuntu-gpu-h200x1-141gb-atl1, resolves to a loopback address: 127.0.1.1; using 10.50.0.5 instead (on interface eth0)
26/05/03 00:17:20 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/03 00:17:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Driver memory      : 180g
Shuffle partitions : 96
Master             : local[24]


---
## 2. Data Loading

In [8]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("chethuhn/network-intrusion-dataset")

print("Path to dataset files:", path)

/root/env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 230M/230M [00:02<00:00, 92.8MB/s] 

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/chethuhn/network-intrusion-dataset/versions/1


In [9]:
CSV_PATH = "/root/.cache/kagglehub/datasets/chethuhn/network-intrusion-dataset/versions/1"

df_csv = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(CSV_PATH)
)

print(f"Partitions : {df_csv.rdd.getNumPartitions()}")
print(f"Row count  : {df_csv.count():,}")
print(f"Columns    : {len(df_csv.columns)}")
df_csv.printSchema()

Partitions : 96
Row count  : 2,830,743
Columns    : 79
root
 |--  Destination Port: integer (nullable = true)
 |--  Flow Duration: integer (nullable = true)
 |--  Total Fwd Packets: integer (nullable = true)
 |--  Total Backward Packets: integer (nullable = true)
 |-- Total Length of Fwd Packets: integer (nullable = true)
 |--  Total Length of Bwd Packets: integer (nullable = true)
 |--  Fwd Packet Length Max: integer (nullable = true)
 |--  Fwd Packet Length Min: integer (nullable = true)
 |--  Fwd Packet Length Mean: double (nullable = true)
 |--  Fwd Packet Length Std: double (nullable = true)
 |-- Bwd Packet Length Max: integer (nullable = true)
 |--  Bwd Packet Length Min: integer (nullable = true)
 |--  Bwd Packet Length Mean: double (nullable = true)
 |--  Bwd Packet Length Std: double (nullable = true)
 |-- Flow Bytes/s: double (nullable = true)
 |--  Flow Packets/s: double (nullable = true)
 |--  Flow IAT Mean: double (nullable = true)
 |--  Flow IAT Std: double (nullable = tr

---
## 3. Exploratory Data Analysis

In [10]:
df_csv.limit(5).toPandas().head()

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,22,166,1,1,0,0,0,0,0.0,0.0,...,32,0.000,0.000,0,0,0.0,0.000,0,0,BENIGN
1,60148,83,1,2,0,0,0,0,0.0,0.0,...,32,0.000,0.000,0,0,0.0,0.000,0,0,BENIGN
2,123,99947,1,1,48,48,48,48,48.0,0.0,...,40,0.000,0.000,0,0,0.0,0.000,0,0,BENIGN
3,123,37017,1,1,48,48,48,48,48.0,0.0,...,32,0.000,0.000,0,0,0.0,0.000,0,0,BENIGN
4,0,111161336,147,0,0,0,0,0,0.0,0.0,...,0,1753752.625,2123197.578,4822992,95,9463032.7,2657727.996,13600000,5700287,BENIGN


In [11]:
# Raw label column is the last column
raw_label_col = df_csv.columns[-1]
print(f"Label column: '{raw_label_col}'")
df_csv.select(raw_label_col).distinct().orderBy(raw_label_col).show(truncate=False)

Label column: ' Label'
+--------------------------+
| Label                    |
+--------------------------+
|BENIGN                    |
|Bot                       |
|DDoS                      |
|DoS GoldenEye             |
|DoS Hulk                  |
|DoS Slowhttptest          |
|DoS slowloris             |
|FTP-Patator               |
|Heartbleed                |
|Infiltration              |
|PortScan                  |
|SSH-Patator               |
|Web Attack � Brute Force  |
|Web Attack � Sql Injection|
|Web Attack � XSS          |
+--------------------------+



In [12]:
df_csv.groupBy(raw_label_col).count().orderBy("count", ascending=False).show(truncate=False)

+--------------------------+-------+
| Label                    |count  |
+--------------------------+-------+
|BENIGN                    |2273097|
|DoS Hulk                  |231073 |
|PortScan                  |158930 |
|DDoS                      |128027 |
|DoS GoldenEye             |10293  |
|FTP-Patator               |7938   |
|SSH-Patator               |5897   |
|DoS slowloris             |5796   |
|DoS Slowhttptest          |5499   |
|Bot                       |1966   |
|Web Attack � Brute Force  |1507   |
|Web Attack � XSS          |652    |
|Infiltration              |36     |
|Web Attack � Sql Injection|21     |
|Heartbleed                |11     |
+--------------------------+-------+



---
## 4. Data Cleaning

Each sub-step refines the DataFrame in place. Checkpoints are written after key milestones.

### 4.1 Strip Leading/Trailing Whitespace from Column Names

CICFlowMeter exports column names with leading spaces — strip them so downstream joins and references work correctly.

In [13]:
cleaned_cols = [c.strip() for c in df_csv.columns]
df_clean = df_csv.toDF(*cleaned_cols)

print("Columns before strip (first 3):", [df_csv.columns[i] for i in range(3)])
print("Columns after  strip (first 3):", cleaned_cols[:3])
df_clean.printSchema()

Columns before strip (first 3): [' Destination Port', ' Flow Duration', ' Total Fwd Packets']
Columns after  strip (first 3): ['Destination Port', 'Flow Duration', 'Total Fwd Packets']
root
 |-- Destination Port: integer (nullable = true)
 |-- Flow Duration: integer (nullable = true)
 |-- Total Fwd Packets: integer (nullable = true)
 |-- Total Backward Packets: integer (nullable = true)
 |-- Total Length of Fwd Packets: integer (nullable = true)
 |-- Total Length of Bwd Packets: integer (nullable = true)
 |-- Fwd Packet Length Max: integer (nullable = true)
 |-- Fwd Packet Length Min: integer (nullable = true)
 |-- Fwd Packet Length Mean: double (nullable = true)
 |-- Fwd Packet Length Std: double (nullable = true)
 |-- Bwd Packet Length Max: integer (nullable = true)
 |-- Bwd Packet Length Min: integer (nullable = true)
 |-- Bwd Packet Length Mean: double (nullable = true)
 |-- Bwd Packet Length Std: double (nullable = true)
 |-- Flow Bytes/s: double (nullable = true)
 |-- Flow Packet

### 4.2 Remove Duplicate Rows

In [14]:
rows_before = df_clean.count()
df_clean = df_clean.dropDuplicates()
rows_after = df_clean.count()

print(f"Rows before dedup : {rows_before:,}")
print(f"Rows after  dedup : {rows_after:,}")
print(f"Duplicates removed: {rows_before - rows_after:,}")

[Stage 17:========================================>              (71 + 24) / 96]

Rows before dedup : 2,830,743
Rows after  dedup : 2,522,362
Duplicates removed: 308,381


In [15]:
# ── Checkpoint 1 — after deduplication ──────────────────────────────────────
CKPT1 = "data/checkpoints/01_dedup"
df_clean.write.mode("overwrite").parquet(CKPT1)

ckpt1_meta = {
    "step": "01_dedup",
    "rows": rows_after,
    "columns": df_clean.columns,
    "path": CKPT1,
}
os.makedirs("data/checkpoints", exist_ok=True)
with open("data/checkpoints/pipeline_step1.pkl", "wb") as f:
    pickle.dump(ckpt1_meta, f)

print(f"Checkpoint saved  → {CKPT1}/")
print(f"Pipeline pkl saved → data/checkpoints/pipeline_step1.pkl")
print("To reload: df_clean = spark.read.parquet('data/checkpoints/01_dedup')")

[Stage 23:========================================>              (70 + 24) / 96]

Checkpoint saved  → data/checkpoints/01_dedup/
Pipeline pkl saved → data/checkpoints/pipeline_step1.pkl
To reload: df_clean = spark.read.parquet('data/checkpoints/01_dedup')


### 4.3 Audit Missing Values

In [17]:
null_counts = df_clean.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in df_clean.columns
]).collect()[0]

cols_with_nulls = {c: null_counts[c] for c in df_clean.columns if null_counts[c] > 0}

print("Null counts per column:")
print("-" * 45)
if cols_with_nulls:
    for c, n in cols_with_nulls.items():
        print(f"  {c:40s}: {n:,}")
else:
    print("  No nulls found.")

[Stage 32:=====================================================>  (92 + 4) / 96]

Null counts per column:
---------------------------------------------
  No nulls found.


### 4.4 Replace Infinity and NaN Values

`Flow Bytes/s` and `Flow Packets/s` contain `Inf` when flow duration is 0. Replace with `null`, then fill with the column mean.

In [18]:
float_cols = [
    f.name for f in df_clean.schema.fields
    if isinstance(f.dataType, (DoubleType, FloatType))
]

# Replace Inf / NaN with null
for c in float_cols:
    df_clean = df_clean.withColumn(
        c,
        when(
            (col(c) == float("inf")) | (col(c) == float("-inf")) | isnan(col(c)),
            None
        ).otherwise(col(c))
    )
# Fill remaining nulls with column mean
means = df_clean.select([F.mean(col(c)).alias(c) for c in float_cols]).collect()[0]
fill_map = {c: float(means[c]) for c in float_cols if means[c] is not None}
df_clean = df_clean.fillna(fill_map)

print(f"Processed {len(float_cols)} float/double columns for Inf/NaN.")

Processed 24 float/double columns for Inf/NaN.


### 4.5 Fix Label Encoding

CICFlowMeter encodes the `–` separator (e.g. *Web Attack – XSS*) as multi-byte UTF-8 sequences that corrupt when CSV is read with the wrong locale. Strip all non-ASCII bytes, then collapse duplicate spaces.

In [19]:
df_clean = df_clean.withColumn(
    "Label",
    trim(
        regexp_replace(
            regexp_replace(col("Label"), "[^\\x00-\\x7F]", ""),  # remove non-ASCII
            "\\s+", " "                                            # collapse spaces
        )
    )
)

print("Distinct labels after encoding fix:")
df_clean.select("Label").distinct().orderBy("Label").show(truncate=False)

Distinct labels after encoding fix:


[Stage 42:==============================================>        (81 + 15) / 96]

+------------------------+
|Label                   |
+------------------------+
|BENIGN                  |
|Bot                     |
|DDoS                    |
|DoS GoldenEye           |
|DoS Hulk                |
|DoS Slowhttptest        |
|DoS slowloris           |
|FTP-Patator             |
|Heartbleed              |
|Infiltration            |
|PortScan                |
|SSH-Patator             |
|Web Attack Brute Force  |
|Web Attack Sql Injection|
|Web Attack XSS          |
+------------------------+



### 4.6 Drop Irrelevant / Redundant Columns

| Reason | Columns |
|--------|---------|
| Zero-variance flag columns | `Fwd/Bwd PSH Flags`, `Fwd/Bwd URG Flags`, `URG/CWE/ECE Flag Count` |
| Broken CICFlowMeter bulk features (all zero) | `Fwd/Bwd Avg Bytes/Packets/Rate Bulk` |
| Redundant (identical to totals) | `Subflow Fwd/Bwd Packets/Bytes` |
| Weak / noisy | `Init_Win_bytes_forward/backward` |

In [20]:
DROP_COLS = [
    # Zero-variance flags
    "Fwd PSH Flags",
    "Bwd PSH Flags",
    "Fwd URG Flags",
    "Bwd URG Flags",
    "URG Flag Count",
    "CWE Flag Count",
    "ECE Flag Count",
    # Broken bulk features
    "Fwd Avg Bytes/Bulk",
    "Fwd Avg Packets/Bulk",
    "Fwd Avg Bulk Rate",
    "Bwd Avg Bytes/Bulk",
    "Bwd Avg Packets/Bulk",
    "Bwd Avg Bulk Rate",
    # Redundant subflow columns
    "Subflow Fwd Packets",
    "Subflow Fwd Bytes",
    "Subflow Bwd Packets",
    "Subflow Bwd Bytes",
    # Weak / noisy
    "Init_Win_bytes_forward",
    "Init_Win_bytes_backward",
]

cols_to_drop = [c for c in DROP_COLS if c in df_clean.columns]
df_clean = df_clean.drop(*cols_to_drop)

print(f"Dropped  : {len(cols_to_drop)} columns")
print(f"Remaining: {len(df_clean.columns)} columns")
print(f"Skipped (not found): {set(DROP_COLS) - set(cols_to_drop)}")

Dropped  : 19 columns
Remaining: 60 columns
Skipped (not found): set()


### 4.7 Standardize Label Names

Map 15 original attack categories into 9 consolidated classes for balanced modelling.

| Original | Consolidated |
|----------|--------------|
| BENIGN | Normal |
| DoS Hulk, DoS GoldenEye, DoS slowloris, DoS Slowhttptest, DDoS | DoS/DDoS |
| FTP-Patator, SSH-Patator | BruteForce |
| Web Attack Brute Force, Web Attack XSS, Web Attack Sql Injection | WebAttack |
| Bot | Botnet |
| PortScan | Recon |
| Infiltration | Infiltration |
| Heartbleed | Heartbleed |

In [21]:
df_clean = df_clean.withColumn(
    "Label",
    when(col("Label") == "BENIGN",                    "Normal")
    .when(col("Label") == "DoS Hulk",                 "DoS/DDoS")
    .when(col("Label") == "DoS GoldenEye",            "DoS/DDoS")
    .when(col("Label") == "DoS slowloris",            "DoS/DDoS")
    .when(col("Label") == "DoS Slowhttptest",         "DoS/DDoS")
    .when(col("Label") == "DDoS",                     "DoS/DDoS")
    .when(col("Label") == "FTP-Patator",              "BruteForce")
    .when(col("Label") == "SSH-Patator",              "BruteForce")
    .when(col("Label") == "Web Attack Brute Force",   "WebAttack")
    .when(col("Label") == "Web Attack XSS",           "WebAttack")
    .when(col("Label") == "Web Attack Sql Injection", "WebAttack")
    .when(col("Label") == "Bot",                      "Botnet")
    .when(col("Label") == "PortScan",                 "Recon")
    .when(col("Label") == "Infiltration",             "Infiltration")
    .when(col("Label") == "Heartbleed",               "Heartbleed")
    .otherwise("Unknown")
)

In [22]:
# Verify no rows fell through to 'Unknown'
unknown_count = df_clean.filter(col("Label") == "Unknown").count()
if unknown_count > 0:
    print(f"WARNING: {unknown_count:,} rows mapped to 'Unknown' — check label mapping!")
    df_clean.filter(col("Label") == "Unknown").select("Label").distinct().show(truncate=False)
else:
    print("Label mapping complete — 0 Unknown rows.")

print("\nDistinct consolidated labels:")
df_clean.select("Label").distinct().orderBy("Label").show(truncate=False)

print("Class distribution after consolidation:")
df_clean.groupBy("Label").count().orderBy("count", ascending=False).show()

Label mapping complete — 0 Unknown rows.

Distinct consolidated labels:


+------------+
|Label       |
+------------+
|Botnet      |
|BruteForce  |
|DoS/DDoS    |
|Heartbleed  |
|Infiltration|
|Normal      |
|Recon       |
|WebAttack   |
+------------+

Class distribution after consolidation:


[Stage 56:==========================================>            (75 + 21) / 96]

+------------+-------+
|       Label|  count|
+------------+-------+
|      Normal|2096484|
|    DoS/DDoS| 321764|
|       Recon|  90819|
|  BruteForce|   9152|
|   WebAttack|   2143|
|      Botnet|   1953|
|Infiltration|     36|
|  Heartbleed|     11|
+------------+-------+



In [23]:
# ── Checkpoint 2 — after full cleaning ──────────────────────────────────────
CKPT2 = "data/checkpoints/02_cleaned"
df_clean.write.mode("overwrite").parquet(CKPT2)

ckpt2_meta = {
    "step": "02_cleaned",
    "rows": df_clean.count(),
    "columns": df_clean.columns,
    "label_col": "Label",
    "label_classes": [
        r["Label"]
        for r in df_clean.select("Label").distinct().orderBy("Label").collect()
    ],
    "dropped_cols": cols_to_drop,
    "path": CKPT2,
}
with open("data/checkpoints/pipeline_step2.pkl", "wb") as f:
    pickle.dump(ckpt2_meta, f)

print(f"Checkpoint saved  → {CKPT2}/")
print(f"Pipeline pkl saved → data/checkpoints/pipeline_step2.pkl")
print("To reload: df_clean = spark.read.parquet('data/checkpoints/02_cleaned')")

[Stage 71:======================================================> (93 + 3) / 96]

Checkpoint saved  → data/checkpoints/02_cleaned/
Pipeline pkl saved → data/checkpoints/pipeline_step2.pkl
To reload: df_clean = spark.read.parquet('data/checkpoints/02_cleaned')


---
## 5. Class Balancing

The dataset is heavily skewed (BENIGN ≈ 80 %). Apply combined under- and over-sampling so each class approaches a target count.

| Class | Target |
|-------|--------|
| Normal | 30 000 |
| DoS/DDoS | 30 000 |
| Recon | 30 000 |
| BruteForce | 30 000 |
| WebAttack | 30 000 |
| Botnet | 30 000 |
| Infiltration | 5 000 |
| Heartbleed | 5 000 |

In [24]:
TARGETS = {
    "Botnet":       30_000,
    "BruteForce":   30_000,
    "DoS/DDoS":     30_000,
    "Heartbleed":    5_000,
    "Infiltration":  5_000,
    "Normal":       30_000,
    "Recon":        30_000,
    "WebAttack":    30_000,
}

# ── 1. Compute actual counts (single scan) ───────────────────────────────────
total_counts = df_clean.groupBy("Label").count()

target_df = spark.createDataFrame(
    [(label, target) for label, target in TARGETS.items()],
    ["Label", "target_count"]
)

fractions_df = (
    total_counts
    .join(target_df, on="Label", how="left")
    .withColumn("target_count", F.coalesce(F.col("target_count"), F.col("count")))
    .withColumn("fraction", F.least(F.lit(1.0), F.col("target_count") / F.col("count")))
)

# ── 2. Undersample in one pass with sampleBy ─────────────────────────────────
fraction_map = {
    row["Label"]: float(row["fraction"])
    for row in fractions_df.select("Label", "fraction").collect()
}

undersampled = df_clean.sampleBy("Label", fractions=fraction_map, seed=42)
undersampled.cache()

# ── 3. Oversample minority classes ───────────────────────────────────────────
needs_oversample = (
    fractions_df
    .filter(F.col("target_count") > F.col("count"))
    .select("Label", "count", "target_count")
    .collect()
)

oversampled_parts = []
for row in needs_oversample:
    label      = row["Label"]
    count      = int(row["count"])
    target     = int(row["target_count"])
    class_df   = df_clean.filter(F.col("Label") == label)

    multiplier = target // count
    remainder  = target % count

    parts = [class_df] * multiplier
    if remainder > 0:
        parts.append(class_df.sample(fraction=remainder / count, seed=42))

    oversampled_class = parts[0]
    for p in parts[1:]:
        oversampled_class = oversampled_class.union(p)

    oversampled_class = oversampled_class.localCheckpoint()  # cut lineage
    oversampled_parts.append(oversampled_class)

# ── 4. Combine all parts ─────────────────────────────────────────────────────
if oversampled_parts:
    oversampled_df = oversampled_parts[0]
    for p in oversampled_parts[1:]:
        oversampled_df = oversampled_df.union(p)
    balanced_df = undersampled.union(oversampled_df)
else:
    balanced_df = undersampled

balanced_df = balanced_df.repartition(200).cache()

print("Balanced class distribution:")
balanced_df.groupBy("Label").count().orderBy("count", ascending=False).show()

Balanced class distribution:


[Stage 129:================================================>   (677 + 24) / 728]

+------------+-----+
|       Label|count|
+------------+-----+
|  BruteForce|39192|
|   WebAttack|32141|
|      Botnet|31987|
|      Normal|30177|
|       Recon|30093|
|    DoS/DDoS|30075|
|Infiltration| 5037|
|  Heartbleed| 5010|
+------------+-----+



In [25]:
# ── Checkpoint 3 — after class balancing ────────────────────────────────────
CKPT3 = "data/checkpoints/03_balanced"
balanced_df.write.mode("overwrite").parquet(CKPT3)

balanced_counts = {
    row["Label"]: row["count"]
    for row in balanced_df.groupBy("Label").count().collect()
}

ckpt3_meta = {
    "step": "03_balanced",
    "total_rows": sum(balanced_counts.values()),
    "class_counts": balanced_counts,
    "targets": TARGETS,
    "seed": 42,
    "path": CKPT3,
}
with open("data/checkpoints/pipeline_step3.pkl", "wb") as f:
    pickle.dump(ckpt3_meta, f)

print(f"Checkpoint saved  → {CKPT3}/")
print(f"Pipeline pkl saved → data/checkpoints/pipeline_step3.pkl")
print("To reload: balanced_df = spark.read.parquet('data/checkpoints/03_balanced')")

Checkpoint saved  → data/checkpoints/03_balanced/
Pipeline pkl saved → data/checkpoints/pipeline_step3.pkl
To reload: balanced_df = spark.read.parquet('data/checkpoints/03_balanced')


---
## 6. Save Final Dataset

Write the balanced, cleaned DataFrame to `data/clean/` in Parquet format.  
This is the file read by the **feature-engineering** notebook.

> **Why Parquet, not `.pkl`?**  
> PySpark DataFrames live in the JVM — Python's `pickle` cannot serialise them.  
> Parquet is the standard portable checkpoint format for Spark: columnar, compressed, and schema-preserving.  
> The `.pkl` files saved above store only the *metadata* (column lists, class counts, paths) so the pipeline configuration can be inspected without starting Spark.

In [ ]:
FINAL_PATH = "data/clean"
balanced_df.write.mode("overwrite").parquet(FINAL_PATH)

feature_cols = [c for c in balanced_df.columns if c != "Label"]

pipeline_final = {
    "step": "final",
    "path": FINAL_PATH,
    "total_rows": sum(balanced_counts.values()),
    "label_col": "Label",
    "feature_cols": feature_cols,
    "num_features": len(feature_cols),
    "class_counts": balanced_counts,
    "label_classes": sorted(balanced_counts.keys()),
    "checkpoints": {
        "01_dedup":   "data/checkpoints/01_dedup",
        "02_cleaned": "data/checkpoints/02_cleaned",
        "03_balanced": "data/checkpoints/03_balanced",
    },
}
os.makedirs("models", exist_ok=True)
with open("models/pipeline_final.pkl", "wb") as f:
    pickle.dump(pipeline_final, f)

print(f"Final dataset saved → {FINAL_PATH}/")
print(f"Pipeline pkl saved  → models/pipeline_final.pkl")
print(f"Features : {len(feature_cols)}")
print(f"Total rows: {sum(balanced_counts.values()):,}")

---
## 7. Cleanup

In [ ]:
spark.stop()
print("SparkSession stopped.")